In [1]:
pip install pandas mysql-connector-python


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import mysql.connector
import pandas as pd
from datetime import datetime

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Root@123#",
    database="HospitalManagement"
)

print("--Database Connected successfully--")


--Database Connected successfully--


In [36]:
class TransparencyLogger:
    @staticmethod
    def log(agent, message):
        print(f"[{agent}] → {message}")


In [37]:
class DataRetrievalAgent:
    def __init__(self, conn):
        self.conn = conn

    def fetch_data(self):
        TransparencyLogger.log("DataRetrievalAgent", "Fetching appointment, doctor, department data")

        query = """
        SELECT 
            a.AppointmentID,
            a.PatientID,
            a.DoctorID,
            a.AppointmentDate,
            d.DepartmentID
        FROM Appointment a
        JOIN Doctor doc ON a.DoctorID = doc.DoctorID
        JOIN Department d ON doc.DepartmentID = d.DepartmentID
        """

        df = pd.read_sql(query, self.conn)

        TransparencyLogger.log(
            "DataRetrievalAgent",
            f"Fetched {len(df)} records with columns {list(df.columns)}"
        )

        return df


In [38]:
class AdmissionCounterAgent:
    def process(self, df):
        TransparencyLogger.log(
            "AdmissionCounterAgent",
            "Counting total appointments per patient"
        )

        grouped = (
            df.groupby("PatientID")
            .size()
            .reset_index(name="TotalAppointments")
        )

        TransparencyLogger.log(
            "AdmissionCounterAgent",
            f"Generated admission counts for {len(grouped)} patients"
        )

        return grouped


In [39]:
class ReadmissionAgent:
    def process(self, admission_df):
        TransparencyLogger.log(
            "ReadmissionAgent",
            "Applying rule: Readmissions = TotalAppointments - 1"
        )

        admission_df["Readmissions"] = admission_df["TotalAppointments"] - 1
        admission_df["Readmissions"] = admission_df["Readmissions"].clip(lower=0)

        for _, row in admission_df.head(5).iterrows():
            TransparencyLogger.log(
                "ReadmissionAgent",
                f"Patient {row.PatientID}: "
                f"{row.TotalAppointments} appointments → "
                f"{row.Readmissions} readmissions"
            )

        return admission_df


In [40]:
class DepartmentAttributionAgent:
    def __init__(self, conn):
        self.conn = conn

    def process(self, readmission_df):
        TransparencyLogger.log(
            "DepartmentAttributionAgent",
            "Mapping patients to department names"
        )

        query = """
        SELECT 
            a.PatientID,
            d.DepartmentID,
            d.DepartmentName
        FROM Appointment a
        JOIN Doctor doc ON a.DoctorID = doc.DoctorID
        JOIN Department d ON doc.DepartmentID = d.DepartmentID
        """

        dept_map = pd.read_sql(query, self.conn)

        merged = pd.merge(dept_map, readmission_df, on="PatientID")

        department_summary = (
            merged.groupby(["DepartmentID", "DepartmentName"])["Readmissions"]
            .sum()
            .reset_index(name="TotalReadmissions")
        )

        return department_summary


In [41]:
class InsightExplanationAgent:
    def process(self, patient_df, department_df):
        TransparencyLogger.log(
            "InsightExplanationAgent",
            "Generating patient and department-level insights"
        )

        # Patient-level insights
        print("\n PATIENT READMISSION SUMMARY\n")
        top_patients = patient_df.sort_values(
            "Readmissions", ascending=False
        ).head(5)
        print(top_patients)

        # Department-level insights
        print("\n🏥 DEPARTMENT READMISSION SUMMARY\n")
        ranked_departments = department_df.sort_values(
            "TotalReadmissions", ascending=False
        )

        for _, row in ranked_departments.iterrows():
            print(
                f"{row.DepartmentName} department handled "
                f"{row.TotalReadmissions} readmissions."
            )



In [42]:
class OrchestratorAgent:
    def __init__(self, conn):
        self.conn = conn

    def run(self):
        TransparencyLogger.log(
            "OrchestratorAgent",
            "Starting swarm intelligence workflow"
        )

        data_agent = DataRetrievalAgent(self.conn)
        raw_data = data_agent.fetch_data()

        admission_agent = AdmissionCounterAgent()
        admission_data = admission_agent.process(raw_data)

        readmission_agent = ReadmissionAgent()
        readmission_data = readmission_agent.process(admission_data)

        department_agent = DepartmentAttributionAgent(self.conn)
        department_data = department_agent.process(readmission_data)

        # DepartmentInsightAgent REMOVED
        # Insights handled here
        insight_agent = InsightExplanationAgent()
        insight_agent.process(readmission_data, department_data)

        TransparencyLogger.log(
            "OrchestratorAgent",
            "Workflow completed successfully"
        )


In [43]:
if __name__ == "__main__":
    orchestrator = OrchestratorAgent(conn)
    orchestrator.run()


[OrchestratorAgent] → Starting swarm intelligence workflow
[DataRetrievalAgent] → Fetching appointment, doctor, department data
[DataRetrievalAgent] → Fetched 1000 records with columns ['AppointmentID', 'PatientID', 'DoctorID', 'AppointmentDate', 'DepartmentID']
[AdmissionCounterAgent] → Counting total appointments per patient
[AdmissionCounterAgent] → Generated admission counts for 628 patients
[ReadmissionAgent] → Applying rule: Readmissions = TotalAppointments - 1
[ReadmissionAgent] → Patient 1: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 2: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 4: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 6: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 8: 1 appointments → 0 readmissions
[DepartmentAttributionAgent] → Mapping patients to department names
[InsightExplanationAgent] → Generating patient and department-level insights

 PATIENT READMISSION SUMMARY

     PatientID  TotalAppointment

C:\Users\sonka\AppData\Local\Temp\ipykernel_10144\1300786032.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.conn)
C:\Users\sonka\AppData\Local\Temp\ipykernel_10144\3828183226.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dept_map = pd.read_sql(query, self.conn)
